In [ ]:
import importlib
import os
import pickle
from pathlib import Path

import gnss_tools.signals.gps_l1ca as gps_l1ca
import gnss_tools.signals.gps_l2c as gps_l2c
import matplotlib.pyplot as plt
import numpy as np
import scipy.signal
import utils.bpsk_correlation as bpsk_correlation

import utils
from utils import bpsk_acquisition, collect_metadata_utils, sample_streaming, tracking_l2c_aligned, tracking_bpsk_aligned

plt.rcParams.update({'font.size': 16})

In [ ]:
# Define signal parameters for each potential signal we might want to test
GPS_L1CA_tracking_signal_params = {
    "G{:02d}".format(prn): tracking_bpsk_aligned.TrackingSignalParameters(
        code_seq=1 - 2 * gps_l1ca.get_GPS_L1CA_code_sequence(prn),
        nominal_code_rate_chips_per_sec=gps_l1ca.CODE_RATE,
        carrier_freq_hz=gps_l1ca.CARRIER_FREQ,
    ) for prn in range(1, 33)
}
# GPS_L2C_tracking_signal_params = {
#     f"G{prn:02d}": tracking_l2c_aligned.L2CTrackingSignalParameters(
#         code_seq_0=1 - 2 * gps_l2c.get_GPS_L2CM_code_sequence(prn),
#         code_seq_1=1 - 2 * gps_l2c.get_GPS_L2CL_code_sequence(prn),
#         nominal_code_rate_chips_per_sec=gps_l2c.CODE_RATE_L2CLM,
#         carrier_freq_hz=gps_l2c.CARRIER_FREQ,
#     )
#     for prn in range(1, 33)
# }
GPS_L2C_tracking_signal_params = {
    f"G{prn:02d}": tracking_bpsk_aligned.TrackingSignalParameters(
        code_seq=1 - 2 * gps_l2c.get_GPS_L2CM_code_sequence(prn),
        nominal_code_rate_chips_per_sec=gps_l2c.CODE_RATE_L2CM,
        carrier_freq_hz=gps_l2c.CARRIER_FREQ,
    )
    for prn in range(1, 33)
}

In [ ]:
# Note: you will need to change the filepath to point to where your data is stored
#  I have mine stored in a "local-data" folder within the project directory
local_data_dir = Path(utils.__file__).parent.parent / "local-data"
collects_dir = local_data_dir / "collects"
available_experiment_names = sorted(map(lambda fp: fp.name, collects_dir.iterdir()))
print("Available experiments:", ", ".join([str(name) for name in available_experiment_names]))
experiment_name = available_experiment_names[2]
experiment_dir = collects_dir / experiment_name
# Note: I keep a metadata.yml file in each experiment directory to keep track of sample metadata
#  and what are the available collect filenames.  You can either create your own metadata.yml,
#  or modify the code below to directly chose your data filepath and set sample parameters.
metadata_filepath = experiment_dir / "metadata.yml"
metadata = collect_metadata_utils.load_experiment_metadata_from_file(metadata_filepath, print_summary=True)

# L1
# collect_id = metadata.collect_ids[2]
# band_id = metadata.band_ids[0]
# sig_params_dict = GPS_L1CA_tracking_signal_params
# L2
collect_id = metadata.collect_ids[1]
band_id = metadata.band_ids[1]
sig_params_dict = GPS_L2C_tracking_signal_params

collect_config = metadata.collects[collect_id]
channel_id = collect_config.channel_config_id
band_config = metadata.band_configurations[band_id]
channel_config = metadata.channel_configurations[channel_id]

inter_freq_hz = band_config.inter_freq
samp_rate = channel_config.samp_rate
sample_params = channel_config.sample_params
collect_filepath = experiment_dir / collect_config.filename

In [ ]:
# Load acquisition results from file
acq_results_directory = local_data_dir / "acquisition-results"
acq_results_directory.mkdir(parents=True, exist_ok=True)
acq_results_version_id = "v1"
acq_results_filepath = acq_results_directory / f"{collect_id}.{acq_results_version_id}.pkl"
with open(acq_results_filepath, "rb") as f:
    acq_results: dict[str, bpsk_acquisition.AcquisitionResult] = pickle.load(f)

acquired_signal_ids = sorted(list(filter(
    lambda sig_id: acq_results[sig_id].signal_detected, acq_results.keys()
)))
print(f"Acquired signals in collect {collect_id}: ")
print(", ".join(acquired_signal_ids))

In [ ]:
buffer_duration_ms = 500
buffer_size_samples = int(samp_rate * buffer_duration_ms / 1e3)

sig_id = acquired_signal_ids[0]
# sig_id = "G01"
sig_params = sig_params_dict[sig_id]
print(f"Example acquired signal ID: {sig_id}")
print(f"Acquisition results for signal {sig_id}:")
acq_result = acq_results[sig_id]
print(f"  Signal detected: {acq_result.signal_detected}")
print(f"  Acquisition code phase (ms): {acq_result.acq_code_phase_seconds * 1e3:.3f}")
print(f"  Acquisition Doppler (Hz): {acq_result.acq_doppler_hz:.2f}")

correlator = tracking_bpsk_aligned.AlignedCorrelator(
    tracking_bpsk_aligned.DelayDopplerCorrelatorConfig(
        3, -0.5, 0.5, 1, 0.0, 0.0
    )
)
acq_signal_state = tracking_bpsk_aligned.TrackingSignalState(
    uptime_epoch_ms=acq_result.uptime_epoch_ms,
    code_phase_ms=acq_result.acq_code_phase_seconds * 1e3,
    code_rate_ms_per_sec=(1.0 + acq_result.acq_doppler_hz / sig_params.carrier_freq_hz) * 1e3,
    carrier_phase_cycles=0.0,
    carrier_rate_cyc_per_sec=acq_result.acq_doppler_hz,
)
corr_interval = tracking_bpsk_aligned.CorrelationInterval(
    int(acq_signal_state.code_phase_ms + 1),
    1
)

In [ ]:
# Performed aligned correlations based on acquisition results
num_correlations = 200
correlations = np.zeros(num_correlations, dtype=complex)
corr_uptimes_ms = np.zeros(num_correlations)

with sample_streaming.FileSampleStream(
    collect_filepath,
    sample_params,
    buffer_size_samples,
 ) as sample_stream:

    sample_buffer_generator = sample_stream.sample_buffer_generator()
    buffer_samples = next(sample_buffer_generator)
    buffer_uptime_epoch_ms = 0.0
    sample_buffer = sample_streaming.SampleBuffer(buffer_samples, buffer_uptime_epoch_ms, samp_rate)
    print(f"Sample buffer uptime epoch: {sample_buffer.start_uptime_ms:.2f} ms")
    
    for i in range(num_correlations):
        approx_corr_start_uptime_ms, approx_corr_stop_uptime_ms = corr_interval.compute_start_and_stop_uptime_ms(acq_signal_state)
        corr_start_buffer_sample_index = int(approx_corr_start_uptime_ms * 1e-3 * samp_rate)
        corr_stop_buffer_sample_index = int(approx_corr_stop_uptime_ms * 1e-3 * samp_rate)
        actual_corr_start_uptime_ms = corr_start_buffer_sample_index / samp_rate * 1e3
        actual_corr_stop_uptime_ms = corr_stop_buffer_sample_index / samp_rate * 1e3
        corr_start_code_phase_ms, corr_start_carr_phase_cyc = acq_signal_state.propagate_phase(actual_corr_start_uptime_ms)

        corr_doppler_hz = acq_signal_state.carrier_rate_cyc_per_sec
        corr_code_phase_chips = corr_start_code_phase_ms / 1e3 * sig_params.nominal_code_rate_chips_per_sec
        corr_code_rate_chips_per_sec = acq_signal_state.code_rate_ms_per_sec / 1e3 * sig_params.nominal_code_rate_chips_per_sec
        
        # print(f"Corr. interval {i}: {actual_corr_start_uptime_ms:.2f} ms to {actual_corr_stop_uptime_ms:.2f} ms, code phase {corr_code_phase_chips:.2f} chips, doppler {corr_doppler_hz:.2f} Hz")
        # print(f"Corr. interval {i}: {corr_start_uptime_ms:.2f} ms to {corr_stop_uptime_ms:.2f} ms")
        # correlator.accumulate(
        #     sample_buffer,
        #     accum_start_uptime_ms=corr_start_uptime_ms,
        #     accum_stop_uptime_ms=corr_stop_uptime_ms,
        #     signal_params=sig_params,
        #     signal_state=acq_signal_state,
        # )
        # correlations[i] = correlator.corr_grid[1, 0]
        corr_result = np.zeros(1, dtype=complex)
        bpsk_correlation.correlate__delay(
                sample_buffer.samples[corr_start_buffer_sample_index:corr_stop_buffer_sample_index],
                sample_buffer.samp_rate,
                corr_start_carr_phase_cyc,
                corr_doppler_hz,
                sig_params.code_seq,
                sig_params.code_length_chips,
                corr_code_rate_chips_per_sec,
                corr_code_phase_chips,
                1,
                0.0,
                0.0,
                corr_result,
            )
        # print(i, corr_stop_buffer_sample_index - corr_start_buffer_sample_index)
        correlations[i] = corr_result[0]
        corr_uptimes_ms[i] = actual_corr_start_uptime_ms

        corr_interval.increment()

unwrapped_corr_phase_cyc = np.unwrap(np.angle(correlations)) / (2 * np.pi)
approx_excess_doppler_hz = (unwrapped_corr_phase_cyc[-1] - unwrapped_corr_phase_cyc[0]) / (corr_uptimes_ms[-1] - corr_uptimes_ms[0]) * 1e3
print(f"Approx. excess Doppler from correlation phase: {approx_excess_doppler_hz:.2f} Hz")

excess_doppler_phase_trend_cyc = np.polyval(np.polyfit(corr_uptimes_ms * 1e-3, unwrapped_corr_phase_cyc, 1), corr_uptimes_ms * 1e-3)

In [ ]:
fig = plt.figure(figsize=(10, 6))
ax = fig.add_subplot(1, 1, 1)
ax.plot(corr_uptimes_ms, np.abs(correlations), marker="o", color="#111111")
ax.scatter(corr_uptimes_ms, correlations.real, color="r")
ax.scatter(corr_uptimes_ms, correlations.imag, color="b")
ax2 = ax.twinx()    
ax2.plot(corr_uptimes_ms, unwrapped_corr_phase_cyc - excess_doppler_phase_trend_cyc, marker="o", color="#aaaaaa")
ax.grid()
# ax.set_ylim(-20e3, 20e3)
ax.set_xlabel("Time [ms]")
ax.set_ylabel("Correlation Magnitude")
ax2.set_ylabel("Correlation Phase [cycles]")
plt.show()

In [ ]:
num_correlations = 1023 * 40
correlations = np.zeros(num_correlations, dtype=complex)
corr_phase_offset_ms = np.zeros(num_correlations)

with sample_streaming.FileSampleStream(
    collect_filepath,
    sample_params,
    buffer_size_samples,
 ) as sample_stream:

    sample_buffer_generator = sample_stream.sample_buffer_generator()
    buffer_samples = next(sample_buffer_generator)
    buffer_uptime_epoch_ms = 0.0
    sample_buffer = sample_streaming.SampleBuffer(buffer_samples, buffer_uptime_epoch_ms, samp_rate)

    for i in range(num_correlations):

        corr_phase_offset_ms[i] = i / sig_params.nominal_code_rate_chips_per_sec * 1e3
        corr_code_phase_ms = acq_result.acq_code_phase_seconds * 1e3 + corr_phase_offset_ms[i]

        corr_carr_phase_cycles = acq_signal_state.carrier_phase_cycles
        corr_doppler_hz = acq_signal_state.carrier_rate_cyc_per_sec
        corr_code_phase_chips = corr_code_phase_ms / 1e3 * sig_params.nominal_code_rate_chips_per_sec
        corr_code_rate_chips_per_sec = acq_signal_state.code_rate_ms_per_sec / 1e3 * sig_params.nominal_code_rate_chips_per_sec
        
        corr_result = np.zeros(1, dtype=complex)
        bpsk_correlation.correlate__delay(
                sample_buffer.samples[0:50000],
                sample_buffer.samp_rate,
                corr_carr_phase_cycles,
                corr_doppler_hz,
                sig_params.code_seq,
                sig_params.code_length_chips,
                corr_code_rate_chips_per_sec,
                corr_code_phase_chips,
                1,
                0.0,
                0.0,
                corr_result,
            )
        # print(i, corr_stop_buffer_sample_index - corr_start_buffer_sample_index)
        correlations[i] = corr_result[0]

In [ ]:
fig = plt.figure(figsize=(10, 6))
ax = fig.add_subplot(1, 1, 1)
ax.plot(corr_phase_offset_ms, np.abs(correlations), marker="o", color="#111111", alpha=0.1)
ax.scatter(corr_phase_offset_ms, correlations.real, color="r")
ax.scatter(corr_phase_offset_ms, correlations.imag, color="b")
ax2 = ax.twinx()
# ax2.plot(corr_phase_offset_ms, np.unwrap(np.angle(correlations)), marker="o", color="#aaaaaa")
ax.grid()
# ax.set_ylim(-20e3, 20e3)
ax.set_xlabel("Time [ms]")
ax.set_ylabel("Correlation Magnitude")
ax2.set_ylabel("Correlation Phase [cycles]")
# ax.set_xlim(.9, 1.1)
# ax.set_xlim(3.55, 3.65)
plt.show()